<a href="https://colab.research.google.com/github/Biegle-W/google_colab_llm/blob/claude%2Fkind-volta-wlb7e5/llm_hub_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab LLM hub: one OpenAI-compatible endpoint for your local harness

Runs llama.cpp and vLLM on a Colab GPU behind **one authenticated OpenAI-compatible API**, like the local model hub in your `local-llm-public` setup:
a request that names a model downloads it (once), starts the right engine, and switches models on demand without cutting off replies.

- **Models live on Google Drive** (`MyDrive/llm_models`), so they are downloaded once, not every session.
- **The models offered are the ones that fit this GPU.** An L4 (24 GB), A100 (40/80 GB) or G4 (about 96 GB) can run much larger models than a pair of 24 GB cards, with no CPU offload.
- **Your harness connects** with a base URL and an API key (a stable key, kept on Drive). The cell at the end prints a ready-made dsh provider block.

The hub code lives in this repository (`colab_llm_hub/`) and is fetched fresh each run, so you never need to reopen the notebook to get fixes.
Runtime -> Change runtime type -> **L4, A100 or G4 GPU**, then run the cells in order.

## 1. Mount Drive and fetch the hub code

In [ ]:
import os, re, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

HUB_REPO  = "https://github.com/Biegle-W/google_colab_llm"
HUB_REF   = None        # a branch, tag or commit SHA to pin; None = the repository's default branch
DRIVE_DIR = "/content/drive/MyDrive/llm_models"
HUB_DIR   = "/content/hub_repo"

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"{cmd}\n{r.stdout}{r.stderr}")
    return r.stdout.strip()

sh(f"rm -rf {HUB_DIR}")
if HUB_REF and re.fullmatch(r"[0-9a-f]{40}", HUB_REF):          # a commit SHA has to be fetched by hash
    sh(f"mkdir -p {HUB_DIR} && cd {HUB_DIR} && git init -q && git fetch -q --depth 1 {HUB_REPO} {HUB_REF} && git checkout -q FETCH_HEAD")
else:
    sh(f"git clone -q --depth 1 {'--branch ' + HUB_REF if HUB_REF else ''} {HUB_REPO} {HUB_DIR}")
sys.path.insert(0, HUB_DIR)
import colab_llm_hub
print("hub", colab_llm_hub.__version__, "| commit", sh(f"cd {HUB_DIR} && git rev-parse --short HEAD"))

## 2. Detect the GPU and see which models fit it

In [ ]:
from colab_llm_hub import catalog, gpu

G = gpu.detect()[0]
VRAM = G["vram_gb"]
ENTRIES = catalog.load()
print(f"{G['name']}: {VRAM} GB VRAM, compute capability {G['compute_cap']} (sm_{G['arch']})")
!free -h | head -2
!df -h /content | tail -1
print()
print(f"{'model id':32s} {'engine':6s} {'size':>7s}  {'context here':>12s}")
for e in ENTRIES:
    fit = catalog.fits(e, VRAM)
    ctx = f"{catalog.context_for(e, VRAM):,}" if fit else f"needs {catalog.min_vram_gb(e)} GB"
    print(f"{e['id']:32s} {e['backend']:6s} {e['size_gb']:6.1f}G  {ctx:>12s}" + ("" if fit else "   (does not fit)"))
if not catalog.visible(ENTRIES, VRAM):
    print("\nNo catalog model fits this GPU. Use an L4, A100 or G4 runtime (for a T4 see llama_cpp_t4.ipynb).")

## 3. Config

Context sizes in the catalog are **estimates for this amount of VRAM, not validated allocations**; override them if a model runs out of memory (or you want more).

In [ ]:
# Models to download to Drive now. The hub also fetches any listed model the first time a request names it,
# but a big download in the middle of a request is slow, so prefetching the ones you use is better.
PREFETCH = ["qwen3.8-27b-q4"]

ENGINES = ["llama"]            # add "vllm" to also serve the vLLM models (installs vLLM; several GB, cached on Drive)
CTX_OVERRIDE = {}              # e.g. {"qwen3.8-27b-q4": 32768}

LLAMA_CPP_REF = "4c3c5810f87efe177949f1b85ae509d38a2a311f"   # the commit the reference setup's flags were checked against; "latest" for the newest
REBUILD_LLAMA = False          # True = ignore the cached build and rebuild
REFRESH_WHEELS = False         # True = re-download vLLM's packages and re-save them on Drive

TUNNEL = "quick"               # "quick": free, random URL every session | "named": stable URL (see the notes at the end)
PORT = 8000

unknown = [m for m in PREFETCH if m not in {e["id"] for e in ENTRIES}]
assert not unknown, f"unknown model ids: {unknown}"
not_fit = [m for m in PREFETCH if not catalog.fits(catalog.by_id(ENTRIES)[m], VRAM)]
assert not not_fit, f"these do not fit this GPU: {not_fit}"
needed = {catalog.by_id(ENTRIES)[m]["backend"] for m in PREFETCH}
assert needed <= set(ENGINES), f"PREFETCH needs engines {sorted(needed - set(ENGINES))}; add them to ENGINES"

## 4. Install the engines (built or downloaded once, then restored from Drive)

In [ ]:
from colab_llm_hub import install

BUILDS = f"{DRIVE_DIR}/builds"
LLAMA_BIN = "llama-server"
if "llama" in ENGINES:
    LLAMA_BIN = install.ensure_llama(BUILDS, G["arch"], ref=LLAMA_CPP_REF, rebuild=REBUILD_LLAMA)
    print("llama-server:", LLAMA_BIN)
if "vllm" in ENGINES:
    print("vLLM", install.ensure_vllm(BUILDS, refresh=REFRESH_WHEELS))

## 5. Download the models to Google Drive (skipped for anything already saved)

In [ ]:
%pip install -q -U huggingface_hub hf_xet
import threading, time
from colab_llm_hub.store import ModelStore

try:                                    # a Hugging Face token (Colab secret HF_TOKEN) is only needed for gated models
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

store = ModelStore(DRIVE_DIR, "/content/models")
for mid in PREFETCH:
    e = catalog.by_id(ENTRIES)[mid]
    if store.state(e)["on_drive"]:
        print(f"{mid}: already on Drive")
        continue
    stop = threading.Event()
    def report():
        while not stop.wait(15):
            p = store.progress.get(mid, {})
            print(f"  {mid}: {p.get('phase', '')} {p.get('done_bytes', 0) / 1e9:.1f} / {e['size_gb']:.1f} GB")
    threading.Thread(target=report, daemon=True).start()
    t0 = time.time()
    try:
        store.prefetch(e)
    finally:
        stop.set()
    print(f"{mid}: saved to Drive in {(time.time() - t0) / 60:.1f} min")

## 6. Start the hub and open the tunnel

Re-running this cell restarts the hub. The API key is created once and kept on Drive, so your harness settings stay valid across sessions.

In [ ]:
import json, requests, shutil, time
from colab_llm_hub import keyfile, tunnel, provider

subprocess.run("pkill -TERM -f '[c]olab_llm_hub serve'; pkill -f '[c]loudflared'", shell=True)   # stop a previous run of this cell ([c] stops the pattern matching its own shell)
time.sleep(3)

API_KEY = keyfile.load_or_create(f"{DRIVE_DIR}/hub/api_key")
args = [sys.executable, "-m", "colab_llm_hub", "serve", "--drive-dir", DRIVE_DIR, "--port", str(PORT),
        "--llama-bin", LLAMA_BIN, "--engines", ",".join(ENGINES)]
for k, v in CTX_OVERRIDE.items():
    args += ["--ctx", f"{k}={v}"]
hub_proc = subprocess.Popen(args, cwd=HUB_DIR, env=dict(os.environ, PYTHONPATH=HUB_DIR),
                            stdout=open("/content/hub.log", "w"), stderr=subprocess.STDOUT)

AUTH = {"Authorization": f"Bearer {API_KEY}"}
for _ in range(60):
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", headers=AUTH, timeout=2).status_code == 200:
            break
    except requests.RequestException:
        pass
    if hub_proc.poll() is not None:
        raise RuntimeError("The hub exited:\n" + open("/content/hub.log").read()[-3000:])
    time.sleep(1)
else:
    raise RuntimeError("The hub did not start:\n" + open("/content/hub.log").read()[-3000:])

cf = tunnel.ensure_cloudflared()
if TUNNEL == "named":
    from google.colab import userdata
    tunnel_proc = tunnel.start_named(userdata.get("CLOUDFLARE_TUNNEL_TOKEN"))
    BASE = "https://" + userdata.get("CLOUDFLARE_HOSTNAME").removeprefix("https://").strip("/")
else:
    tunnel_proc, BASE = tunnel.start_quick(PORT)

MODELS = requests.get(f"http://127.0.0.1:{PORT}/v1/models", headers=AUTH).json()["data"]
json.dump({"base_url": BASE + "/v1", "updated": time.strftime("%Y-%m-%d %H:%M:%S"), "gpu": G["name"],
           "models": [m["id"] for m in MODELS]}, open(f"{DRIVE_DIR}/hub/endpoint.json", "w"), indent=1)

print("Base URL :", BASE + "/v1")
print("API key  :", API_KEY)
print(f"Models   : {', '.join(m['id'] for m in MODELS)}")
print()
print("Any OpenAI-compatible client:")
print(provider.openai_env(BASE, API_KEY))
print("dsh: on your computer, in a clone of this repo, this adds the provider to its settings for you:")
print(f"  python -m colab_llm_hub.configure_dsh --url {BASE} --key {API_KEY} --settings <path to dsh settings.yaml>")
print("or merge this block by hand, and set the key in the environment where dsh runs:")
print(f"  export COLAB_LLM_API_KEY={API_KEY}\n")
print(provider.dsh_provider_yaml(BASE, MODELS))

## 7. Try it from here (the first request to a model loads it, so it can take a while)

In [ ]:
MODEL = PREFETCH[0] if PREFETCH else MODELS[0]["id"]
t0 = time.time()
r = requests.post(f"http://127.0.0.1:{PORT}/v1/chat/completions", headers=AUTH, timeout=900,
                  json={"model": MODEL, "max_tokens": 200, "messages": [{"role": "user", "content": "In two sentences, what is a model hub?"}]})
if r.status_code == 503 and r.json().get("error", {}).get("code") == "model_loading":
    print("The model is still loading; run this cell again in a minute. (Streaming requests wait for it automatically.)")
else:
    j = r.json()
    print(j["choices"][0]["message"]["content"] if r.ok else j)
    print(f"\n[{time.time() - t0:.1f} s, usage: {j.get('usage')}]")

## 8. Status and logs (re-run any time)

In [ ]:
st = requests.get(f"http://127.0.0.1:{PORT}/v1/hub/status", headers=AUTH).json()
print(json.dumps({k: st[k] for k in ("gpu", "vram_gb", "active", "loading", "phase", "inflight", "errors", "local_free_gb")}, indent=1))
for mid, p in st["downloads"].items():
    print(f"  {mid}: {p.get('phase')} {p.get('done_bytes', 0) / 1e9:.1f} / {p.get('total_bytes', 0) / 1e9:.1f} GB")
print("\n--- hub log ---"); print(open("/content/hub.log").read()[-1500:])
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 9. Stop

In [ ]:
hub_proc.terminate()        # stops the hub and the engine it started
tunnel_proc.terminate()

## Notes

- **Use from your local harness:** point it at the Base URL with the API key, then pick any model id from `/v1/models`. Naming a model that is not loaded makes the hub load it (a streaming request is kept alive while it loads; a non-streaming one gets `503` + `Retry-After` after about 85 s while the load continues, so retry it).
- **One model at a time** is loaded. A switch waits for replies in progress to finish, then stops the old engine and starts the new one.
- **Stable URL.** A quick tunnel gets a new random URL every session. For a fixed one, create a free Cloudflare **named tunnel** (Zero Trust -> Networks -> Tunnels), add a public hostname that points to `http://localhost:8000`, and save its token as the Colab secret `CLOUDFLARE_TUNNEL_TOKEN` and the hostname as `CLOUDFLARE_HOSTNAME`; then set `TUNNEL = "named"`. The current URL is also written to `MyDrive/llm_models/hub/endpoint.json`.
- **Security.** The URL is public; only the API key protects it. Anyone with both can use your GPU. The key file is on your Drive; delete it to rotate the key.
- **Models** are listed in `colab_llm_hub/catalog.json`; add your own (any GGUF repo for llama.cpp, any safetensors repo for vLLM). Contexts there are estimates, and the vLLM entries (especially the experimental Nemotron one) have not been validated on this vLLM version.
- **Speed and quality are not benchmarked here.** Nothing in this notebook has been measured on a Colab GPU yet.
- Logs: `/content/hub.log` (hub), `/content/hub_logs/` (engines), `/content/install.log` (builds).